# Scrape and Analyze a Live Website

Companion notebook for the course project [Scrape and Analyze a Live Website](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/scrape-analyze).

This notebook scrapes real, live quotes from [quotes.toscrape.com](https://quotes.toscrape.com) — a public site built and maintained specifically for scraping practice — follows its pagination, cleans the result with pandas, and produces two charts.

**No API key, no free-tier signup, nothing to configure** — just a real website and the Python scraping/data stack. That makes this project a genuinely good fit for Colab/Kaggle/Binder: no local file server, no GPU, no long-running process to manage, and inline chart output is exactly what a notebook does well.

See the [full lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/scrape-analyze) for the step-by-step walkthrough and the reasoning behind each step.

## Step 0: Install dependencies

Run this once per notebook session (Colab/Kaggle/Binder all support `!pip install` cells).

In [ ]:
!pip install requests beautifulsoup4 pandas matplotlib


## Step 1: Fetch and parse one page

`quotes.toscrape.com` has no login wall, no rate limiting to fight, and a stable HTML layout — quotes live in `<div class="quote">`, with text in `<span class="text">`, author in `<small class="author">`, and tags in `<a class="tag">`.

> Always check `robots.txt` and terms of service before scraping any site other than `quotes.toscrape.com` — this lesson deliberately uses a site built for scraping practice.

In [ ]:
import time

import requests
from bs4 import BeautifulSoup

response = requests.get("https://quotes.toscrape.com/", timeout=10)
response.raise_for_status()  # turns a 404/500 into a loud exception instead of a silent bad parse
soup = BeautifulSoup(response.text, "html.parser")

for quote_div in soup.find_all("div", class_="quote"):
    text = quote_div.find("span", class_="text").get_text(strip=True)
    author = quote_div.find("small", class_="author").get_text(strip=True)
    tags = [tag.get_text(strip=True) for tag in quote_div.find_all("a", class_="tag")]
    print(f"{author}: {text} {tags}")


## Step 2: Handle pagination and collect all the data

Rather than hardcode "loop 10 times," follow the "Next" link itself — that way the scraper keeps working even if the number of pages changes. A short `time.sleep()` between requests is standard scraping etiquette, even on a practice-friendly site.

In [ ]:
import csv

BASE_URL = "https://quotes.toscrape.com"
REQUEST_DELAY_SECONDS = 1.0


def parse_quotes(soup):
    """Extracts {"text", "author", "tags"} for every quote on one parsed page."""
    quotes = []
    for quote_div in soup.find_all("div", class_="quote"):
        text = quote_div.find("span", class_="text").get_text(strip=True)
        author = quote_div.find("small", class_="author").get_text(strip=True)
        tags = [t.get_text(strip=True) for t in quote_div.find_all("a", class_="tag")]
        quotes.append({"text": text, "author": author, "tags": ", ".join(tags)})
    return quotes


def scrape_all_quotes():
    all_quotes = []
    url = f"{BASE_URL}/"
    page_number = 1

    while url is not None:
        print(f"Fetching page {page_number}: {url}")
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
        except requests.RequestException as exc:
            # One failed request shouldn't kill a scrape that's already collected
            # data from several pages -- log it and stop cleanly instead of crashing.
            print(f"  Failed to fetch {url}: {exc}. Stopping here.")
            break

        page_soup = BeautifulSoup(resp.text, "html.parser")
        page_quotes = parse_quotes(page_soup)
        print(f"  Found {len(page_quotes)} quotes.")
        all_quotes.extend(page_quotes)

        next_li = page_soup.find("li", class_="next")
        url = requests.compat.urljoin(url, next_li.find("a")["href"]) if next_li else None
        page_number += 1
        if url is not None:
            time.sleep(REQUEST_DELAY_SECONDS)

    return all_quotes


quotes = scrape_all_quotes()
print(f"\nScraped {len(quotes)} quotes total.")


In [ ]:
with open("quotes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["text", "author", "tags"])
    writer.writeheader()
    writer.writerows(quotes)

print("Saved quotes.csv to the notebook's working directory.")


## Step 3: Load into pandas and clean

`tags` was saved as a single `"tag1, tag2, tag3"` string, because CSV cells can't hold a real Python list — reconstruct it with `.apply`. Whitespace and dtype checks are cheap to do and easy to skip, and the kind of thing that silently breaks a `groupby` later if left unchecked.

In [ ]:
import pandas as pd

df = pd.read_csv("quotes.csv")

# tags was saved as a single "tag1, tag2, tag3" string -- split it into a real
# list column so it can be exploded into one row per tag below.
df["tags"] = df["tags"].fillna("").apply(
    lambda raw: [tag.strip() for tag in raw.split(",") if tag.strip()]
)

# Whitespace and dtype sanity checks.
df["text"] = df["text"].str.strip()
df["author"] = df["author"].str.strip()
assert df["text"].notna().all(), "some quotes have no text -- check the scrape"
assert df["author"].notna().all(), "some quotes have no author -- check the scrape"

df["quote_length"] = df["text"].str.len()

print(f"Loaded {len(df)} quotes")
print(f"Unique authors: {df['author'].nunique()}")
print(f"Average quote length: {df['quote_length'].mean():.0f} characters")
df.head()


## Step 4: Analyze and visualize

`%matplotlib inline` makes charts render directly below each cell — works the same way in Colab, Kaggle, and Binder.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt


**Most common tags** — `explode` turns the list-of-tags column into one row per tag, so `value_counts` can count them individually.

In [ ]:
BAR_COLOR = "#3b82f6"  # one consistent hue -- this is magnitude, not identity, so one color is correct

exploded = df.explode("tags")
exploded = exploded[exploded["tags"] != ""]
tag_counts = exploded["tags"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
tag_counts.sort_values().plot(kind="barh", ax=ax, color=BAR_COLOR)
ax.set_xlabel("Number of quotes")
ax.set_ylabel("Tag")
ax.set_title(f"Top {len(tag_counts)} tags on quotes.toscrape.com")
ax.set_xlim(left=0)  # bar charts must start at 0 -- truncating exaggerates differences
fig.tight_layout()
fig.savefig("top_tags.png")
plt.show()


**Most-quoted authors:**

In [ ]:
most_quoted = df["author"].value_counts().head(5)
most_quoted


**Quote-length distribution** — a histogram, to see the shape of the data rather than just a single average.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["quote_length"], bins=20, color=BAR_COLOR, edgecolor="white")
ax.set_xlabel("Quote length (characters)")
ax.set_ylabel("Number of quotes")
ax.set_title("Distribution of quote lengths")
fig.tight_layout()
fig.savefig("quote_length_dist.png")
plt.show()


## What you just built

A complete, honest fetch → parse → clean → analyze → visualize pipeline, running against a real live website instead of a file someone else prepared for you — entirely inside a hosted notebook, with no API key and no local setup.

See the [full lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/scrape-analyze) for common pitfalls, Socratic questions, and where to go from here.